# 02.有监督筛选

计算变量总体 KS/AUC、分月 KS，并输出有监督筛选结果。

In [ ]:
import os
import sys
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
SRC_DIR = PROJECT_ROOT / "src"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(SRC_DIR))

from 函数代码包 import *

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: "%.6f" % x)

customer_type = "demo_existing_customer"
target_type = "y2"   # 可切换为 "y1"

join_keys = ["id", "cell", "name", "apply_date"]
keep_cols = ["month", "flag"]
target = "y"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("customer_type:", customer_type)
print("target_type:", target_type)

In [ ]:
data_df = pd.read_csv(OUTPUT_DIR / f"1.建模宽表_{customer_type}_{target_type}.csv")
data_df = data_df.fillna(-999)

exclude_cols = list(set(join_keys + keep_cols + [target, "y1", "y2"]))
var_cols = [c for c in data_df.columns if c not in exclude_cols]

print(data_df.shape, len(var_cols))

In [ ]:
# 类别变量编码
cat_cols = [c for c in var_cols if str(data_df[c].dtype) in ["object", "category"]]

for col in cat_cols:
    data_df[col] = data_df[col].astype(str).astype("category").cat.codes

ks_all_df = var_ks(data_df, var_cols, flagy=target)
ks_all_df = ks_all_df.rename(columns={"auc": "auc_all", "ks": "ks_all"})

display(ks_all_df.sort_values("ks_all", ascending=False).head())

In [ ]:
monthly_rows = []

for col in var_cols:
    tmp = []
    for m, g in data_df.groupby("month"):
        if g[target].nunique() < 2:
            continue
        try:
            one = var_ks(g, [col], flagy=target)
            tmp.append(one["ks"].iloc[0])
        except Exception:
            pass

    if len(tmp) == 0:
        ks_avg = ks_min = ks_std = ks_cv = ks_range = np.nan
    else:
        arr = np.array(tmp)
        ks_avg = arr.mean()
        ks_min = arr.min()
        ks_std = arr.std()
        ks_cv = ks_std / max(ks_avg, 1e-9)
        ks_range = arr.max() - arr.min()

    monthly_rows.append({
        "var": col,
        "ks_avg": ks_avg,
        "ks_min": ks_min,
        "ks_std": ks_std,
        "ks_cv": ks_cv,
        "ks_range": ks_range
    })

ks_month_df = pd.DataFrame(monthly_rows)

ks_df = ks_all_df.merge(ks_month_df, on="var", how="left")
ks_df.to_csv(OUTPUT_DIR / f"2.KS结果_{customer_type}_{target_type}.csv", index=False)

display(ks_df.sort_values("ks_all", ascending=False).head())